# Regression Polynomiale

Notebook dedie a l'extraction et a la verification des donnees avant la regression polynomiale.


## Vérification des données extraites

Cette cellule affiche les premières lignes, les dimensions et les statistiques de base du tableau extrait.

## Préparation des variables pour la régression

On sépare la variable explicative et la cible pour la régression polynomiale.

In [ ]:
X = extracted[['TyreLife']].copy()
y = extracted['DeltaToBestLapForStint'].copy()

print('X shape:', X.shape)
print('y shape:', y.shape)
display(X.head())
display(y.head())

In [10]:
import numpy as np


def _estimate_lateral_g_from_xy(data):
    required_xy = {'X', 'Y'}
    if not required_xy.issubset(set(data.columns)):
        return np.nan

    if 'SessionTime' in data.columns:
        t = data['SessionTime'].dt.total_seconds().to_numpy(dtype=float)
    elif 'Time' in data.columns:
        t = data['Time'].dt.total_seconds().to_numpy(dtype=float)
    elif 'Date' in data.columns:
        t = (data['Date'] - data['Date'].iloc[0]).dt.total_seconds().to_numpy(dtype=float)
    else:
        return np.nan

    x = data['X'].to_numpy(dtype=float)
    y = data['Y'].to_numpy(dtype=float)

    valid = np.isfinite(t) & np.isfinite(x) & np.isfinite(y)
    t, x, y = t[valid], x[valid], y[valid]

    if len(t) < 5:
        return np.nan

    dt = np.diff(t)
    keep = np.r_[True, dt > 0]
    t, x, y = t[keep], x[keep], y[keep]

    if len(t) < 5:
        return np.nan

    x_dot = np.gradient(x, t)
    y_dot = np.gradient(y, t)
    x_ddot = np.gradient(x_dot, t)
    y_ddot = np.gradient(y_dot, t)

    speed = np.sqrt(x_dot**2 + y_dot**2)
    denom = (x_dot**2 + y_dot**2) ** 1.5

    with np.errstate(divide='ignore', invalid='ignore'):
        curvature = np.abs(x_dot * y_ddot - y_dot * x_ddot) / denom
        a_lat = speed**2 * curvature
        g_lat = a_lat / 9.80665

    g_lat = g_lat[np.isfinite(g_lat)]
    if g_lat.size == 0:
        return np.nan

    return float(np.median(g_lat))


def get_circuit_lateral_stress(session):
    laps = session.laps.pick_quicklaps()

    lateral_stresses = []

    for _, lap in laps.iterlaps():
        # get_pos_data est plus léger que get_telemetry pour ce besoin X/Y.
        pos_data = lap.get_pos_data()
        avg_g_lat = _estimate_lateral_g_from_xy(pos_data)
        if np.isfinite(avg_g_lat):
            lateral_stresses.append(avg_g_lat)

    if not lateral_stresses:
        return np.nan
    return float(np.mean(lateral_stresses))

In [ ]:
# Mapping de précision pour la saison 2024
# Échelle de 1.0 (très faible) à 5.0 (extrême)
# Basé sur les rapports techniques Pirelli et les caractéristiques de l'asphalte
f1_circuit_precision_stats = {
    "Bahrain Grand Prix": {"lat_energy": 3.2, "abrasivity": 4.8, "type": "Traction"},
    "Saudi Arabian Grand Prix": {"lat_energy": 3.9, "abrasivity": 2.1, "type": "High-Speed Street"},
    "Australian Grand Prix": {"lat_energy": 3.1, "abrasivity": 2.8, "type": "Semi-Permanent"},
    "Japanese Grand Prix": {"lat_energy": 5.0, "abrasivity": 4.6, "type": "High-Energy"},
    "Chinese Grand Prix": {"lat_energy": 4.1, "abrasivity": 3.2, "type": "Front-Limited"},
    "Miami Grand Prix": {"lat_energy": 2.9, "abrasivity": 2.7, "type": "Street"},
    "Emilia Romagna Grand Prix": {"lat_energy": 3.4, "abrasivity": 3.1, "type": "Technical"},
    "Monaco Grand Prix": {"lat_energy": 1.2, "abrasivity": 1.1, "type": "Low-Speed Street"},
    "Canadian Grand Prix": {"lat_energy": 1.8, "abrasivity": 2.3, "type": "Stop-and-Go"},
    "Spanish Grand Prix": {"lat_energy": 4.7, "abrasivity": 4.2, "type": "Aero-Reference"},
    "Austrian Grand Prix": {"lat_energy": 3.1, "abrasivity": 2.4, "type": "Short/Fast"},
    "British Grand Prix": {"lat_energy": 4.9, "abrasivity": 3.3, "type": "High-Energy"},
    "Hungarian Grand Prix": {"lat_energy": 2.8, "abrasivity": 2.2, "type": "Twisty"},
    "Belgian Grand Prix": {"lat_energy": 4.6, "abrasivity": 3.8, "type": "Power/Energy"},
    "Dutch Grand Prix": {"lat_energy": 4.4, "abrasivity": 3.1, "type": "Banking"},
    "Italian Grand Prix": {"lat_energy": 2.7, "abrasivity": 2.2, "type": "Ultra-High Speed"},
    "Azerbaijan Grand Prix": {"lat_energy": 2.1, "abrasivity": 2.4, "type": "Street/Straight"},
    "Singapore Grand Prix": {"lat_energy": 1.9, "abrasivity": 4.1, "type": "Street/Bumpy"},
    "United States Grand Prix": {"lat_energy": 4.2, "abrasivity": 4.3, "type": "Technical/Bumpy"},
    "Mexico City Grand Prix": {"lat_energy": 2.3, "abrasivity": 2.1, "type": "Altitude"},
    "São Paulo Grand Prix": {"lat_energy": 3.2, "abrasivity": 3.4, "type": "Technical"},
    "Las Vegas Grand Prix": {"lat_energy": 1.8, "abrasivity": 1.9, "type": "Cold Street"},
    "Qatar Grand Prix": {"lat_energy": 5.0, "abrasivity": 3.9, "type": "Extreme Lateral"},
    "Abu Dhabi Grand Prix": {"lat_energy": 3.1, "abrasivity": 3.1, "type": "Standard"}
}

## Extraction des G latéraux sur 24 GP + estimation de durée

Cette section exécute la fonction sur chaque GP du calendrier et estime le temps total à partir des durées réellement observées.